# Phase 3e — Two more pre-declared external arms: DDTI + TNUI-train (LG-PM external track)

**Why:** Phase 3d showed TNUI-val carries almost no label↔feature association for ANY
TN5000-trained score — including true-mask oracles (AUROC 0.577–0.624) — while TNUI-test
carries a strong one (oracle 0.834–0.851). Only the operating-point result replicated
(pooled P5 −0.120, p<0.0001). Two questions, declared before any arm is evaluated:

- **Arm E (DDTI, independent institution):** does the rank-transfer picture (3c test)
  generalise to a genuinely independent dataset with a meaningful benign count?
- **Arm F (TNUI-2021 train):** is TNUI-val anomalous *within* TNUI (label noise /
  case-mix), or is the whole external TNUI label relationship weak?

**Frozen decision rules (nothing refits on any external data):** identical to 3c/3d —
same asserted K=20 support episode; linear heads from the Phase 3c cache (tolerance
fallback documented); thresholds + temperature from TN5000 val only; ERM sanity gate
0.9401 ± 0.002; replication gate vs the locked 3c table (+ locked 3d val Dice 0.7307
when the val bank is recomputed). Each arm is its own Bonferroni family of 5
(α = 0.05/5 = 0.01): E1–E5 (DDTI), F1–F5 (TNUI-train). TNUI-val is included in the
descriptive cross-cohort evidence matrix only (bank from 3d cache or recompute);
no new hypothesis tests on val here.

**DDTI label protocol (pre-declared):** per-case XML diagnoses; malignant-if-any-rule
(mirrors TN5000); literal "benign"/"malignant" strings override numeric codes;
numeric 1 → benign, 3 → malignant, anything else (incl. 2) → excluded as unknown.
TI-RADS literals: 1/2/3 → benign, 4/4a/4b/4c/5 → malignant. Profile + counts printed.

## 0. Configuration, Phase 3 cache discovery, checkpoint discovery

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT  = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "tnui": AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
    "tn5000": AUTHOR_ROOT / "tn5000" / "TN5000_forReview",
}
P3 = Path("/kaggle/working/phase3_classification")
P3C = Path("/kaggle/working/phase3c_external")
P3D = Path("/kaggle/working/phase3d_external")
OUTPUT_ROOT = Path("/kaggle/working/phase3e_external")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# protocol constants identical to Phases 1-3
IMG_SIZE, MASK_THRESH, PRED_THRESH = 384, 127, 0.5
SEG_K, RING_PX = 20, 15
PROTO_SCALES, SCORE_HW = ["e2", "e3", "e4"], 48
FEAT_SCALES = ["e3", "e4"]
BATCH, N_BOOT = 16, 2000
ALPHA_BONF = 0.05 / 5
LIN_EPOCHS, LIN_PATIENCE, LIN_LR, LIN_WD = 300, 25, 3e-3, 1e-2

ck_enc = list(Path("/kaggle/input").rglob("best_resnet34_seed42.pt")) + \
         list(Path("/kaggle/working").rglob("best_resnet34_seed42.pt"))
ck_dec = list(Path("/kaggle/input").rglob("best_fusion_decoder.pt")) + \
         list(Path("/kaggle/working").rglob("best_fusion_decoder.pt"))
assert ck_enc and ck_dec, "attach the Phase 1/2 checkpoint Kaggle datasets"
CKPT_ENC, CKPT_DEC = ck_enc[0], ck_dec[0]

NEED = ["features_train.npz", "features_val.npz", "features_test.npz",
        "linear_heads_results.json", "erm_resnet34_tn5000.pt",
        "erm_resnet34_probs.json", "run_summary.json"]
miss = [f for f in NEED if not (P3 / f).exists()]
assert not miss, f"missing Phase 3 caches {miss} - run Phase 3 first in this working dir"
print("device:", DEVICE)
DDTI_ROOT = PATHS["tn3k"].parent.parent / "DDTI dataset" / "DDTI"
DDTI_XML_ROOTS = [DDTI_ROOT, AUTHOR_ROOT / "tn3k-data",
                  Path("/kaggle/input/datasets/dasmehdixtr/ddti-thyroid-ultrasound-images")]
print("caches:", P3, "| 3c:", P3C, P3C.exists(), "| 3d:", P3D, P3D.exists(),
      "| output:", OUTPUT_ROOT)
print("ddti root:", DDTI_ROOT, DDTI_ROOT.exists())

## 1. TNUI-2021 test manifest with labels; TN5000 cross-dataset duplicate audit

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
import re as _re_ddti
re = _re_ddti

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def build_tnui_split(split):
    imgs = {p.stem: p for p in image_files(PATHS["tnui"]/split/"images")}
    msks = {p.stem: p for p in image_files(PATHS["tnui"]/split/"masks")}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"id": k, "image": str(imgs[k]), "mask": str(msks[k])}
                         for k in common]), common

def load_tnui_labels_split(root, split, common):
    rows = {}
    for cf in sorted(Path(root).rglob("*labels*.csv")):
        if split not in cf.stem.lower():
            continue
        try:
            df = pd.read_csv(cf)
        except Exception:
            continue
        stem_col = None; lab_col = None
        for c in df.columns:
            vals = df[c].astype(str).str.strip().str.lower()
            if lab_col is None and set(vals.unique()) <= {"0", "1", "benign", "malignant"}:
                lab_col = c
        for c in df.columns:
            if c == lab_col:
                continue
            vals = df[c].astype(str).map(lambda s: Path(s).stem)
            if sum(v in common for v in vals) > max(10, int(0.5 * len(common))):
                stem_col = c
        if stem_col and lab_col:
            for _, r in df.iterrows():
                k = Path(str(r[stem_col])).stem
                v = str(r[lab_col]).strip().lower()
                if k in common and v in {"0", "1", "benign", "malignant"}:
                    rows[k] = 1 if v in {"1", "malignant"} else 0
    return rows

def attach_labels(df, lab_map, name):
    n_lab = df.id.map(lab_map).notna().sum()
    df = df[df.id.map(lab_map).notna()].copy()
    df["label"] = df.id.map(lab_map).astype(int)
    df = df.reset_index(drop=True)
    print(f"TNUI {name}: labels matched {n_lab} | usable n={len(df)} | "
          f"malignant={int(df.label.sum())} ({df.label.mean():.3f}) | benign={int((1-df.label).sum())}")
    return df

# ---------------- DDTI: images+masks (1_or_data layout) + XML diagnoses ----------------
import xml.etree.ElementTree as ET

def ddti_xml_label(path):
    """Return 1 malignant / 0 benign / None unknown. Element-aware under ElementTree;
    raw-text fallback if parsing fails. All evidence votes; malignant-if-any aggregation
    (mirrors TN5000). Diagnosis codes: 1 -> 0, 3 -> 1, else unknown. TI-RADS 1-3 -> 0,
    4/4a/4b/4c/5 -> 1."""
    pairs, raw = [], ""
    try:
        root = ET.parse(path).getroot()
        pairs = [(str(e.tag).lower(), (e.text or "").strip().lower()) for e in root.iter()]
    except Exception:
        try:
            raw = Path(path).read_text(errors="ignore").lower()
        except Exception:
            return None
    votes = []
    def add_code(t):
        if t.isdigit():
            v = int(t)
            if v == 1:
                votes.append(0)
            elif v == 3:
                votes.append(1)
    if pairs:
        texts = " ".join(t for _, t in pairs if t)
        if "malignant" in texts:
            votes.append(1)
        if "benign" in texts:
            votes.append(0)
        for tag, t in pairs:
            if tag.endswith("diagnosis"):
                add_code(t)
            if ("tirads" in tag or "ti-rads" in tag) and t:
                m = re.match(r"([1-5])[abc]?", t)
                if m:
                    votes.append(1 if m.group(1) in "45" else 0)
        for mm in re.finditer(r"ti-?rads[^0-9]{0,6}([1-5])[abc]?", texts):
            votes.append(1 if mm.group(1) in "45" else 0)
    else:
        if "malignant" in raw:
            votes.append(1)
        if "benign" in raw:
            votes.append(0)
        for mm in re.finditer(r"<diagnosis>\s*([0-9]+)", raw, flags=re.I):
            add_code(mm.group(1))
        for mm in re.finditer(r"ti-?rads[^0-9]{0,6}([1-5])[abc]?", raw):
            votes.append(1 if mm.group(1) in "45" else 0)
    return (1 if 1 in votes else 0) if votes else None

def ddti_case_keys(stem):
    return [stem, stem.split("_")[0]]

ddti_imgs = {p.stem: p for p in image_files(DDTI_ROOT/"1_or_data"/"image")}
ddti_msks = {p.stem: p for p in image_files(DDTI_ROOT/"1_or_data"/"mask")}
ddti_ids = sorted(set(ddti_imgs) & set(ddti_msks))
xmls = []
for r_ in DDTI_XML_ROOTS:
    if Path(r_).exists():
        xmls += sorted(Path(r_).rglob("*.xml"))
xml_lab = {}
for xp in xmls:
    v = ddti_xml_label(xp)
    if v is not None:
        xml_lab[xp.stem] = v
print(f"DDTI pairs found: {len(ddti_ids)} | XMLs parsed (labelled): {len(xml_lab)}/{len(xmls)}")
lab_map_d = {}
for k in ddti_ids:
    for ck in ddti_case_keys(k):
        if ck in xml_lab:
            lab_map_d[k] = xml_lab[ck]; break
ddti = pd.DataFrame([{"id": k, "image": str(ddti_imgs[k]), "mask": str(ddti_msks[k]),
                      "label": lab_map_d[k]} for k in ddti_ids if k in lab_map_d])
if len(ddti) < 100:
    print("!! DDTI label matching low - diagnostics:")
    print("  sample image stems:", ddti_ids[:8])
    print("  sample xml stems:", [p.stem for p in xmls[:8]])
    for r_ in DDTI_XML_ROOTS:
        r_ = Path(r_)
        print(" ", r_, "exists:", r_.exists(),
              "| children:", [c.name for c in list(r_.iterdir())[:8]] if r_.exists() else "-")
    raise AssertionError("DDTI labels need parser adaptation - paste the diagnostics above")
print(f"DDTI usable: n={len(ddti)} | malignant={int(ddti.label.sum())} ({ddti.label.mean():.3f}) "
      f"| benign={int((1-ddti.label).sum())}")

# ---------------- TNUI train / test (gate cohort) ----------------
tnui_train, c_tr = build_tnui_split("train")
tnui_train = attach_labels(tnui_train, load_tnui_labels_split(PATHS["tnui"], "train", set(c_tr)), "train")
tnui_test, c_te = build_tnui_split("test")
tnui_test = attach_labels(tnui_test, load_tnui_labels_split(PATHS["tnui"], "test", set(c_te)), "test")

# ---------------- audits ----------------
AUD = OUTPUT_ROOT / "audit_hashes.json"
if AUD.exists():
    aud = json.loads(AUD.read_text())
else:
    ddti["sha256"] = ddti["image"].apply(sha256_file)
    tr_hash = {sha256_file(p) for p in tnui_train["image"]}
    te_hash = {sha256_file(p) for p in tnui_test["image"]}
    t5_hash = {sha256_file(p) for p in image_files(PATHS["tn5000"]/"JPEGImages")}
    aud = {"ddti_hashes": ddti.set_index("id").sha256.to_dict(),
           "ddti_within_dups": int(ddti.sha256.duplicated().sum()),
           "ddti_vs_tn5000": sorted(set(ddti.sha256) & t5_hash),
           "ddti_vs_tnui_test": sorted(set(ddti.sha256) & te_hash),
           "ddti_vs_tnui_train": sorted(set(ddti.sha256) & tr_hash),
           "tnuitrain_vs_test": sorted(tr_hash & te_hash)}
    AUD.write_text(json.dumps(aud, indent=1))
if "sha256" not in ddti.columns:
    ddti["sha256"] = ddti.id.map(aud["ddti_hashes"])
print(f"audits: ddti within-dups {aud['ddti_within_dups']} (dropped keep-first) | "
      f"ddti vs TN5000 {len(aud['ddti_vs_tn5000'])} | ddti vs TNUI-test {len(aud['ddti_vs_tnui_test'])} | "
      f"ddti vs TNUI-train {len(aud['ddti_vs_tnui_train'])} | TNUI-train vs test {len(aud['tnuitrain_vs_test'])}")
_bad = set(aud["ddti_vs_tn5000"]) | set(aud["ddti_vs_tnui_test"]) | set(aud["ddti_vs_tnui_train"])
if _bad:
    ddti = ddti[~ddti.sha256.isin(_bad)].reset_index(drop=True)
if aud["ddti_within_dups"]:
    ddti = ddti.drop_duplicates("sha256", keep="first").reset_index(drop=True)
print(f"DDTI final n={len(ddti)} | malignant={int(ddti.label.sum())} | benign={int((1-ddti.label).sum())}")

## 2. Loaders (identical to Phase 3) + fixed support episode cross-check

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci, {"s": s, "t": t, "l": l, "nh": nh, "nw": nw}

def letterbox_mask(msk_raw, size=IMG_SIZE):
    h, w = msk_raw.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    rm = cv2.resize(msk_raw, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    cm = np.zeros((size, size), np.uint8); cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return cm

def make_ring(mask, r=RING_PX):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k)
    ring = np.clip(dil.astype(np.int16) - mask.astype(np.int16), 0, 1).astype(np.uint8)
    if ring.sum() == 0:
        ring = (1 - mask.astype(np.uint8))
    return ring

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

# -------- TN3K train pool and THE SAME support episode as Phase 3 --------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    im = {p.stem: p for p in image_files(image_dir)}
    ms = {p.stem: p for p in image_files(mask_dir)}
    cm = sorted(set(im) & set(ms))
    return pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                          "image": str(im[k]), "mask": str(ms[k])} for k in cm])

tn3k = pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval")
tn3k["sha256"] = tn3k["image"].apply(sha256_file)
tn3k = tn3k.sort_values("id", kind="stable").drop_duplicates("sha256", keep="first")
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(tn3k))
val_ids = set(tn3k.iloc[perm[: int(round(0.15 * len(tn3k)))]].id)
tn3k["split"] = np.where(tn3k.id.isin(val_ids), "val", "train")
tn3k_train = tn3k[tn3k.split == "train"].reset_index(drop=True)

def tn3k_support_rows(kmax=SEG_K, seed=SEED + 10):
    r = np.random.RandomState(seed)
    return tn3k_train.iloc[r.permutation(len(tn3k_train))[:kmax]].reset_index(drop=True)

SUP_ROWS = tn3k_support_rows()
_arch = json.loads((P3/"run_summary.json").read_text())["config"]["support_episode_ids"]
assert list(SUP_ROWS.id) == _arch, "support episode mismatch vs Phase 3 archive - seed drift!"
print(f"support episode reproduced exactly ({SEG_K} ids match Phase 3 archive); pool n={len(tn3k_train)}")

def load_tn3k_item(row):
    img = read_gray(row["image"]).astype(np.float32)
    ci, _ = letterbox_image(img)
    cm = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    return ci, cm

## 3. Frozen LG-PM components (exact copies from Phase 3)

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class Phase1UNet(nn.Module):
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e1": e1, "e2": e2, "e3": e3, "e4": e4}

def load_ckpt(path):
    # torch >= 2.6 defaults torch.load(..., weights_only=True); our own checkpoints
    # may embed numpy scalars in metadata (ERM ckpt val-AUROC), which the
    # restricted unpickler rejects. Try the restricted loader first, then fall
    # back explicitly -- files are our own trusted artifacts.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)

backbone = Phase1UNet().to(DEVICE)
_ck = load_ckpt(CKPT_ENC)
backbone.load_state_dict(_ck["model"], strict=True)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def encode(x):
    with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        f = backbone.features(x)
    return {k: v.float() for k, v in f.items()}
print("encoder loaded (val Dice:", round(float(_ck.get("val_dice")), 4), ")")

In [ ]:
def masked_proto(feats, masks):
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    return -(feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)

class FusionDecoder(nn.Module):
    def __init__(self, cin=2 * len(PROTO_SCALES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits_384(que_feats, pf_pb_per_scale, decoder):
    return F.interpolate(decoder(score_maps(que_feats, pf_pb_per_scale)),
                         size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

decoder = FusionDecoder().to(DEVICE)
_dk = load_ckpt(CKPT_DEC)
decoder.load_state_dict(_dk["decoder"], strict=True)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

_sup = [load_tn3k_item(r) for _, r in SUP_ROWS.iterrows()]
sup_x = torch.stack([to_tensor(i) for i, _ in _sup]).to(DEVICE)
sup_m = torch.stack([torch.from_numpy(m[None]) for _, m in _sup]).float().to(DEVICE)
with torch.no_grad():
    SF = encode(sup_x)
PF_PB = {name: masked_proto(SF[name], sup_m) for name in PROTO_SCALES}
print("decoder loaded | fixed support prototypes ready (3 scales)")

## 4. TNUI feature bank — zero-shot segmentation + descriptors + mask QC (Dice)
Oracle rows use **true TNUI masks** (stronger oracle than TN5000’s expert box).

In [ ]:
def dice_np(a, b, eps=1e-7):
    a, b = (a > 0).astype(np.float32), (b > 0).astype(np.float32)
    inter = (a * b).sum()
    return float((2 * inter + eps) / (a.sum() + b.sum() + eps))

def masked_gap(feat, msk01):
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)
    return (feat * m).sum(dim=(2, 3)) / den

REPS = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtmask", "R2g_gtmask_ring"]
def run_cohort_bank(df, tag):
    fpath = OUTPUT_ROOT / f"features_tnui_{tag}.npz"
    if fpath.exists():
        z = np.load(fpath, allow_pickle=True)
        print(f"[cache] TNUI {tag} feature bank loaded")
        return {k: z[k] for k in z.files}
    store = {r: [] for r in REPS}
    mask96, conf, area, empty, ids, labels, dices = [], [], [], [], [], [], []
    t0 = time.time()
    for i0 in range(0, len(df), BATCH):
        chunk = df.iloc[i0:i0 + BATCH]
        imgs, gts = [], []
        for _, r in chunk.iterrows():
            ci, _ = letterbox_image(read_gray(r["image"]).astype(np.float32))
            imgs.append(ci)
            gts.append(letterbox_mask((read_gray(r["mask"]) >= MASK_THRESH).astype(np.uint8)))
        x = torch.stack([to_tensor(i) for i in imgs]).to(DEVICE)
        with torch.no_grad():
            feats = encode(x)
            prob = torch.sigmoid(predict_logits_384(feats, PF_PB, decoder))
        pm = (prob >= PRED_THRESH).float()
        pm_raw = pm.clone()
        for bi in range(len(chunk)):
            if pm[bi].sum() < 4:
                pm[bi] = 1.0; empty.append(1)
            else:
                empty.append(0)
        gtm = torch.from_numpy(np.stack(gts)[:, None]).float().to(DEVICE)
        ring_p = torch.stack([torch.from_numpy(make_ring(m[0].cpu().numpy().astype(np.uint8))[None])
                              for m in pm]).float().to(DEVICE)
        ring_g = torch.stack([torch.from_numpy(make_ring(gts[bi])[None])
                              for bi in range(len(chunk))]).float().to(DEVICE)
        combo = {"R0_image": "image",
                 "R1p_lesion": pm, "R2p_lesion_ring": (pm, ring_p),
                 "R1g_gtmask": gtm, "R2g_gtmask_ring": (gtm, ring_g)}
        desc = {}
        for rep, key in combo.items():
            if key == "image":
                desc[rep] = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1).float().cpu()
            elif isinstance(key, tuple):
                a = torch.cat([masked_gap(feats[nm], key[0]) for nm in FEAT_SCALES], 1)
                b = torch.cat([masked_gap(feats[nm], key[1]) for nm in FEAT_SCALES], 1)
                desc[rep] = torch.cat([a, b], 1).float().cpu()
            else:
                desc[rep] = torch.cat([masked_gap(feats[nm], key) for nm in FEAT_SCALES], 1).float().cpu()
        for r_ in REPS:
            store[r_].append(desc[r_].numpy())
        for bi in range(len(chunk)):
            mask96.append(cv2.resize(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), (96, 96),
                                     interpolation=cv2.INTER_NEAREST))
            conf.append(float(prob[bi].mean())); area.append(float(pm_raw[bi].mean()))
            dices.append(dice_np(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), gts[bi]))
        ids.extend(chunk.id.tolist()); labels.extend(chunk.label.astype(int).tolist())
        done = min(i0 + BATCH, len(df))
        if done % (BATCH * 25) < BATCH or done == len(df):
            print(f"  tnui-{tag}: {done}/{len(df)} images ({(time.time()-t0)/60:.1f} min)")
    out = {"ids": np.array(ids), "labels": np.array(labels, np.int64),
           "mask96": np.stack(mask96), "conf": np.array(conf, np.float32),
           "area": np.array(area, np.float32), "empty": np.array(empty, np.int64),
           "dice": np.array(dices, np.float32)}
    for r_ in REPS:
        out[r_] = np.concatenate(store[r_]).astype(np.float32)
    np.savez_compressed(fpath, **out)
    print(f"[done] tnui-{tag} n={len(ids)} | mean seg Dice {np.mean(dices):.4f} "
          f"(Phase 2 zero-shot K=20 ref 0.708) | empty-mask {sum(empty)} ({100*sum(empty)/len(ids):.1f}%) "
          f"| mean area {np.mean(area):.3f} | {(time.time()-t0)/60:.1f} min")
    return out

# harm: tag -> (df, cache dir, locked-dice assert if recomputed)
def get_bank(tag, df, locked_dice=None):
    for cdir in (P3D, P3C, OUTPUT_ROOT):
        f = cdir / f"features_tnui_{tag}.npz"
        if f.exists():
            z = np.load(f, allow_pickle=True)
            print(f"[cache] {tag} bank from {cdir.name}")
            return {k: z[k] for k in z.files}, f"cache {cdir.name}"
    if tag in ("val", "test"):
        df2 = tnui_val if tag == "val" else tnui_test
        out = run_cohort_bank(df2, tag)
        assert abs(np.mean(out["dice"]) - locked_dice) <= 0.005, \
            f"recomputed {tag} bank deviates from locked Dice {locked_dice} - investigate"
        return out, "recomputed (locked-Dice assert OK)"
    return run_cohort_bank(df, tag), "computed fresh"

tnui_val, _cv = build_tnui_split("val")
tnui_val = attach_labels(tnui_val, load_tnui_labels_split(PATHS["tnui"], "val", set(_cv)), "val")
# exactly the locked-3d val exclusions: val-Test + val-TN5000 overlaps, then within-val dups
tnui_val["sha256"] = tnui_val["image"].apply(sha256_file)
AUD3D = P3D / "audit_hashes.json"
if AUD3D.exists():
    _a3 = json.loads(AUD3D.read_text())
    _badv = set(_a3["overlap_tn5000"]) | set(_a3["overlap_tnui_test"])
else:  # recompute the identical policy (cost: hashing val+test+TN5000 once)
    _teh = {sha256_file(p) for p in tnui_test["image"]}
    _t5h = {sha256_file(p) for p in image_files(PATHS["tn5000"]/"JPEGImages")}
    _badv = (set(tnui_val.sha256) & _teh) | (set(tnui_val.sha256) & _t5h)
tnui_val = tnui_val[~tnui_val.sha256.isin(_badv)].drop_duplicates("sha256", keep="first") \
    .reset_index(drop=True)

DDTI_BANK, SRC_DDTI = get_bank("ddti", ddti)
TRAIN_BANK, SRC_TRAIN = get_bank("tnuitrain", tnui_train)
VAL_BANK, SRC_VAL = get_bank("val", tnui_val, locked_dice=0.7307)
TEST_BANK, SRC_TEST = get_bank("test", tnui_test, locked_dice=0.7046)
BANKS = {"DDTI": DDTI_BANK, "TNUI-train": TRAIN_BANK, "TNUI-val": VAL_BANK, "TNUI-test": TEST_BANK}
# hard alignment gates: bank row order MUST equal df row order for the cached cohorts
DF_ARMS = {"DDTI": ddti, "TNUI-train": tnui_train, "TNUI-val": tnui_val, "TNUI-test": tnui_test}
for _nm in BANKS:
    assert list(BANKS[_nm]["ids"]) == list(DF_ARMS[_nm].id), f"{_nm} bank/df id misalignment"
for nm, b in BANKS.items():
    print(f"{nm:10s} n={len(b['ids']):4d} | dice {np.mean(b['dice']):.4f} | empty {100*np.mean(b['empty']):.1f}%")

## 5. Source-fitted classifiers — retrained linear heads (tolerance-gated) + ERM (sanity-gated)
Linear heads are re-derived from cached TN5000 features under the Phase 3 protocol.
Exact weight reproduction is not promised (RNG after ERM differs) — instead we gate on
val AUROC within 0.01 of the Phase 3 originals and report both. ERM must re-hit
0.9401 ± 0.002 on TN5000 test, else the external row is not trusted.

In [ ]:
P3B = {s: {k: v for k, v in np.load(P3 / f"features_{s}.npz", allow_pickle=True).items()}
       for s in ["train", "val", "test"]}
LIN_ORIG = json.loads((P3/"linear_heads_results.json").read_text())

def feat(rep, split):
    return torch.from_numpy(P3B[split][rep].astype(np.float32))

def labels_np(split):
    return P3B[split]["labels"].astype(np.float32)

def train_linear(rep):
    xtr, xva, xte = feat(rep, "train"), feat(rep, "val"), feat(rep, "test")
    ytr_np = labels_np("train")
    mu, sd = xtr.mean(0, keepdim=True), xtr.std(0, keepdim=True).clamp(min=1e-6)
    xtr, xva, xte = (xtr-mu)/sd, (xva-mu)/sd, (xte-mu)/sd
    xtr, xva, xte = xtr.to(DEVICE), xva.to(DEVICE), xte.to(DEVICE)
    ytr = torch.from_numpy(ytr_np).to(DEVICE)
    lin = nn.Linear(xtr.shape[1], 1).to(DEVICE)
    pos_w = torch.tensor([(len(ytr_np)-ytr_np.sum())/max(ytr_np.sum(), 1)], device=DEVICE)
    opt = torch.optim.AdamW(lin.parameters(), lr=LIN_LR, weight_decay=LIN_WD)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    best, wait, bw = -1, 0, None
    n = len(xtr)
    for ep in range(1, LIN_EPOCHS + 1):
        perm = torch.randperm(n, device=DEVICE)
        lin.train()
        for i0 in range(0, n, 512):
            idx = perm[i0:i0+512]
            loss = bce(lin(xtr[idx]).squeeze(1), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        lin.eval()
        with torch.no_grad():
            pv = torch.sigmoid(lin(xva).squeeze(1)).cpu().numpy()
        va = roc_auc_score(labels_np("val"), pv)
        if va > best + 1e-4:
            best, wait = va, 0
            bw = {k: v.detach().cpu().clone() for k, v in lin.state_dict().items()}
        else:
            wait += 1
            if wait >= LIN_PATIENCE:
                break
    lin.load_state_dict(bw); lin.eval()
    with torch.no_grad():
        out = {s: torch.sigmoid(lin(x).squeeze(1)).cpu().numpy()
               for s, x in [("val", xva), ("test", xte)]}
    head = {"w": bw["weight"].numpy(), "b": bw["bias"].numpy(),
            "mu": mu.numpy(), "sd": sd.numpy()}
    return {"val_auroc": float(best), "probs": out, "head": head}

REPS_SRC = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
MAP_EXT = {"R0_image": "R0_image", "R1p_lesion": "R1p_lesion",
           "R2p_lesion_ring": "R2p_lesion_ring", "R1g_gtbox": "R1g_gtmask",
           "R2g_gtbox_ring": "R2g_gtmask_ring"}
LINW_3C = P3C / "linear_heads_retrained.json"
LINW_PATH = OUTPUT_ROOT / "linear_heads_retrained.json"
HEADS_SRC = None
if LINW_3C.exists():
    LINR = json.loads(LINW_3C.read_text())
    HEADS_SRC = "phase3c cache (identical heads across cohorts - the pre-declaration)"
    print("[cache] 3c heads loaded:", HEADS_SRC)
elif LINW_PATH.exists():
    LINR = json.loads(LINW_PATH.read_text())
    HEADS_SRC = "3d-local retrained cache"
    print("[cache] 3d-local retrained heads loaded")
else:
    HEADS_SRC = "retrained in 3d (tolerance-gated)"
    LINR = {}
    for rep in REPS_SRC:
        LINR[rep] = train_linear(rep)
        o = LIN_ORIG[rep]["val_auroc"]
        d = LINR[rep]["val_auroc"] - o
        flag = "OK" if abs(d) <= 0.01 else "OUT-OF-TOLERANCE"
        print(f"{rep:16s} val AUROC {LINR[rep]['val_auroc']:.4f} (orig {o:.4f}, delta {d:+.4f}) {flag}")
        LINR[rep]["probs"] = {s: v.tolist() for s, v in LINR[rep]["probs"].items()}
        LINR[rep]["head"] = {k: v.tolist() for k, v in LINR[rep]["head"].items()}
    LINW_PATH.write_text(json.dumps(LINR, indent=1))
print("heads provenance:", HEADS_SRC)

def lin_apply(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    z = ((X.astype(np.float64) - mu) / sd) @ w.T + b
    return (1 / (1 + np.exp(-z.ravel()))).astype(float)

# ERM sanity gate
import torchvision.models as tvm
erm = tvm.resnet34(weights=None); erm.fc = nn.Linear(erm.fc.in_features, 1)
erm.load_state_dict(load_ckpt(P3/"erm_resnet34_tn5000.pt")["model"])
erm = erm.to(DEVICE).eval()

class ImgDS(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        ci, _ = letterbox_image(read_gray(self.df.iloc[i]["image"]).astype(np.float32))
        return to_tensor(ci), np.float32(self.df.iloc[i]["label"])

# TN5000 test sanity: rebuild its image rows from cached order? ids are in features npz
# ERM json probs are already aligned with TN5000 splits (saved in Phase 3) -> use them as truth
@torch.no_grad()
def erm_probs(df):
    ld = DataLoader(ImgDS(df), batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)
    ps = []
    for x, _ in ld:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            ps.append(torch.sigmoid(erm(x)).squeeze(1).float().cpu())
    return torch.cat(ps).numpy()

DF_ARMS = {"DDTI": ddti, "TNUI-train": tnui_train, "TNUI-val": tnui_val, "TNUI-test": tnui_test}
P_ERM = {nm: erm_probs(df) for nm, df in DF_ARMS.items()}
print("ERM probs:", {nm: len(p) for nm, p in P_ERM.items()})

## 6. External metrics on TNUI test — source-fitted thresholds and temperature

In [ ]:
def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def fit_temperature(y, p):
    z, yv = logit(np.asarray(p)), np.asarray(y, np.float64)
    grid = np.exp(np.linspace(np.log(0.05), np.log(20.0), 200))
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(yv * np.log(q + 1e-12) + (1 - yv) * np.log(1 - q + 1e-12))
    return float(grid[int(np.argmin([nll(T) for T in grid]))])

def ece(y, p, bins=15):
    y = np.asarray(y, np.float64).ravel(); p = np.asarray(p, np.float64).ravel()
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def brier(y, p):
    return float(np.mean((np.asarray(p).ravel() - np.asarray(y).ravel()) ** 2))

def metric_set(y, p, thr):
    y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
    return {"auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "sens@spec90": float((p[y == 1] >= thr).mean()), "brier": brier(y, p),
            "ece": ece(y, p)}

def boot_ci(y, p, thr, n=N_BOOT, seed=SEED):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    rng = np.random.RandomState(seed); N = len(y); acc = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        acc.append([roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx]),
                    (p[idx][y[idx] == 1] >= thr).mean()])
    lo, hi = np.percentile(acc, 2.5, 0), np.percentile(acc, 97.5, 0)
    return lo, hi

MODELS = ["ERM", "R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtmask", "R2g_gtmask_ring"]

def cohort_probs(bank, p_erm):
    P = {"ERM": np.asarray(p_erm).ravel()}
    for rep_src, rep_ext in MAP_EXT.items():
        P[rep_ext] = lin_apply(rep_src, bank[rep_ext])
    return P

Y = {nm: b["labels"].astype(int) for nm, b in BANKS.items()}
P = {nm: cohort_probs(BANKS[nm], P_ERM[nm]) for nm in BANKS}

# replication gate vs LOCKED 3c (test) and 3d (val, descriptive)
LOCK3C = {"ERM": 0.5172, "R0_image": 0.5791, "R1p_lesion": 0.7119,
          "R2p_lesion_ring": 0.6776, "R1g_gtmask": 0.8343, "R2g_gtmask_ring": 0.8507}
LOCK3D = {"ERM": 0.6237, "R0_image": 0.5901, "R1p_lesion": 0.5775,
          "R2p_lesion_ring": 0.6008, "R1g_gtmask": 0.6126, "R2g_gtmask_ring": 0.5770}
_hetol = 0.0015 if HEADS_SRC.startswith("phase3c cache") else 0.01
drift_t = {m: abs(roc_auc_score(Y["TNUI-test"], P["TNUI-test"][m]) - LOCK3C[m]) for m in MODELS}
drift_v = {m: round(abs(roc_auc_score(Y["TNUI-val"], P["TNUI-val"][m]) - LOCK3D[m]), 4) for m in MODELS}
GATE_OK = all(v <= _hetol for v in drift_t.values())
print("replication vs locked 3c (tol", _hetol, "):", {m: round(v, 4) for m, v in drift_t.items()},
      "-> GATE_OK" if GATE_OK else "-> GATE FAILED: 3c-era rows not trusted here")
print("replication vs locked 3d (val, descriptive):", drift_v)

# source val probs for thresholds/temperature; ERM sanity gate
ERMJ = json.loads((P3/"erm_resnet34_probs.json").read_text())
P_SRCV = {"ERM": np.array(ERMJ["probs"]["val"]).ravel()}
for rep_src in REPS_SRC:
    P_SRCV[MAP_EXT[rep_src]] = np.array(LINR[rep_src]["probs"]["val"]).ravel()
y_srcv = labels_np("val").astype(int)
erm_ref = roc_auc_score(P3B["test"]["labels"].astype(int), np.array(ERMJ["probs"]["test"]).ravel())
assert abs(erm_ref - 0.9401) <= 0.002, f"ERM sanity failed: {erm_ref:.4f}"
print(f"ERM TN5000 test AUROC sanity: {erm_ref:.4f} (ref 0.9401) OK")

THR_EXT, TEMP_EXT = {}, {}
for m in MODELS:
    T = fit_temperature(y_srcv, P_SRCV[m]); TEMP_EXT[m] = T
    thr, _, _ = spec90_threshold(y_srcv, P_SRCV[m]); THR_EXT[m] = thr

def main_table(nm):
    y = Y[nm]
    rows = []
    for m in MODELS:
        pe = P[nm][m]
        pec = 1 / (1 + np.exp(-logit(pe) / TEMP_EXT[m]))
        ms = metric_set(y, pe, THR_EXT[m])
        ms["brier"] = brier(y, pec); ms["ece"] = ece(y, pec)
        lo, hi = boot_ci(y, pe, THR_EXT[m])
        rows.append({"model": m, "auroc": round(ms["auroc"], 4),
                     "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})", "auprc": round(ms["auprc"], 4),
                     "sens@spec90(TN5000val)": round(ms["sens@spec90"], 4),
                     "brier": round(ms["brier"], 4), "ece_post": round(ms["ece"], 4),
                     "T_srcval": round(TEMP_EXT[m], 3)})
    df = pd.DataFrame(rows)
    print(f"{nm} (n={len(y)}, malignant={int(y.sum())}, benign={int((1-y).sum())}) "
          f"- thresholds/temp from TN5000 val:")
    display(df)
    return df

T_E = main_table("DDTI");        T_E.to_csv(OUTPUT_ROOT/"phase3e_ddti_main_table.csv", index=False)
T_F = main_table("TNUI-train");  T_F.to_csv(OUTPUT_ROOT/"phase3e_tnuitrain_main_table.csv", index=False)

## 7. Domain-shift comparison and the five planned paired tests

In [ ]:
def delta_pair_boot(y, pa, pb, n=5000, seed=SEED):
    y = np.asarray(y); rng = np.random.RandomState(seed); N = len(y); d = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2: continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(2*min((d<=0).mean(), (d>=0).mean()), 1.0), len(d)

def paired_boot_sens(y, pa, ta, pb, tb, n=5000, seed=SEED):
    y = np.asarray(y); pos = y == 1
    rng = np.random.RandomState(seed)
    sa = (pa[pos] >= ta).astype(float); sb = (pb[pos] >= tb).astype(float)
    M = len(sa); d = []
    for _ in range(n):
        idx = rng.randint(0, M, M)
        d.append(sa[idx].mean() - sb[idx].mean())
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(2*min((d<=0).mean(), (d>=0).mean()), 1.0), len(d)

def run_family(nm, letter):
    fam = [
        (f"{letter}1 AUROC R1p - R0 (grounding, {nm})", "auroc", "R1p_lesion", "R0_image"),
        (f"{letter}2 AUROC ERM - R1p (fine-tune advantage, {nm})", "auroc", "ERM", "R1p_lesion"),
        (f"{letter}3 AUROC R1g(mask) - R1p (error propagation, {nm})", "auroc", "R1g_gtmask", "R1p_lesion"),
        (f"{letter}4 AUROC ERM - R1g(mask) (fine-tune vs oracle, {nm})", "auroc", "ERM", "R1g_gtmask"),
        (f"{letter}5 sens@spec90 ERM - R1p (operating point, {nm})", "sens", "ERM", "R1p_lesion"),
    ]
    y = Y[nm]; rows = []
    for name, kind, a, b in fam:
        if kind == "auroc":
            m_, lo, hi, p, nb = delta_pair_boot(y, P[nm][a], P[nm][b])
        else:
            m_, lo, hi, p, nb = paired_boot_sens(y, P[nm][a], THR_EXT[a], P[nm][b], THR_EXT[b])
        rows.append({"test": name, "delta(A-B)": round(m_, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                     "p": round(p, 5), "sig(0.01)": "YES" if p < ALPHA_BONF else "no", "n_boot": nb})
    df = pd.DataFrame(rows)
    print(f"{nm} paired tests (Bonferroni alpha = {ALPHA_BONF}, own family of 5):")
    display(df)
    return df

F_E = run_family("DDTI", "E");        F_E.to_csv(OUTPUT_ROOT/"phase3e_ddti_paired.csv", index=False)
F_F = run_family("TNUI-train", "F");  F_F.to_csv(OUTPUT_ROOT/"phase3e_tnuitrain_paired.csv", index=False)

# ---------------- cross-cohort evidence matrix (DESCRIPTIVE, no new tests) ----------------
ERMJ = json.loads((P3/"erm_resnet34_probs.json").read_text())
y_srct = P3B["test"]["labels"].astype(int)
src_test = {"ERM": np.array(ERMJ["probs"]["test"]).ravel()}
for rep_src, rep_ext in MAP_EXT.items():
    src_test[rep_ext] = np.array(LINR[rep_src]["probs"]["test"]).ravel()
mat_auroc, mat_sens = {}, {}
for m in MODELS:
    mat_auroc[m] = {"TN5000-test (in-domain)": round(roc_auc_score(y_srct, src_test[m]), 3)}
    mat_sens[m] = {"TN5000-test (in-domain)": None}
    y = P3B["test"]["labels"].astype(int)
    pv = src_test[m]
    mat_sens[m]["TN5000-test (in-domain)"] = round(float((pv[y == 1] >= THR_EXT[m]).mean()), 3)
    for nm in ["TNUI-test", "TNUI-val", "TNUI-train", "DDTI"]:
        mat_auroc[m][nm] = round(roc_auc_score(Y[nm], P[nm][m]), 3)
        yy = Y[nm]
        mat_sens[m][nm] = round(float((P[nm][m][yy == 1] >= THR_EXT[m]).mean()), 3)
M_A = pd.DataFrame(mat_auroc).T
M_S = pd.DataFrame(mat_sens).T
M_A.to_csv(OUTPUT_ROOT/"phase3e_evidence_auroc.csv")
M_S.to_csv(OUTPUT_ROOT/"phase3e_evidence_sens.csv")
print("evidence matrix - AUROC across cohorts (descriptive):")
display(M_A)
print("evidence matrix - sens@spec90 across cohorts (descriptive):")
display(M_S)
sign_tbl = pd.DataFrame({
    "cohort": ["TNUI-test", "TNUI-val", "TNUI-train", "DDTI"],
    "sign(R1p-ERM AUROC)": [np.sign(roc_auc_score(Y[nm], P[nm]["R1p_lesion"])
                                    - roc_auc_score(Y[nm], P[nm]["ERM"])) for nm in
                            ["TNUI-test", "TNUI-val", "TNUI-train", "DDTI"]],
    "sign(R1p-ERM sens@spec90)": [np.sign((P[nm]["R1p_lesion"][Y[nm] == 1] >= THR_EXT["R1p_lesion"]).mean()
                                          - (P[nm]["ERM"][Y[nm] == 1] >= THR_EXT["ERM"]).mean())
                                  for nm in ["TNUI-test", "TNUI-val", "TNUI-train", "DDTI"]]})
sign_tbl.to_csv(OUTPUT_ROOT/"phase3e_sign_consistency.csv", index=False)
print("sign consistency (descriptive):")
display(sign_tbl)

## 7b. No pooling in 3e — rationale
The pre-declared TNUI test+val pooled sensitivity lives (locked) in Phase 3d. Pooling
DDTI or TNUI-train with TNUI data would mix institutions/label regimes *post hoc*, so this
phase contributes **per-arm families only**. The cross-cohort evidence matrix above is the
descriptive synthesis: it shows where signal exists, per cohort, per model, with frozen
operating points — the input to the paper's externalisation figure.

In [ ]:
print("No pooled analysis in Phase 3e by design (see 7b markdown): per-arm families "
      "E1-E5 / F1-F5 above; pooled TNUI result locked in Phase 3d (P5 sig, P1-P4 null).")

## 8. Qualitative panel — TNUI test: image, true mask, predicted mask+ring, calibrated p(mal)

In [ ]:
def qual_grid(bank, df, nm, seed, picks=(3, 3)):
    rng_q = np.random.RandomState(seed)
    pos_idx = df.index[df.label == 1].to_numpy(); neg_idx = df.index[df.label == 0].to_numpy()
    n_pos, n_neg = min(picks[0], len(pos_idx)), min(picks[1], len(neg_idx))
    pick = list(rng_q.choice(pos_idx, n_pos, replace=False)) + list(rng_q.choice(neg_idx, n_neg, replace=False))
    id2row = {b: i for i, b in enumerate(bank["ids"])}
    p_q = 1/(1+np.exp(-logit(P[nm]["R1p_lesion"])/TEMP_EXT["R1p_lesion"]))
    fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.1 * len(pick)))
    if len(pick) == 1:
        axes = axes[None]
    for r_, i_ in enumerate(pick):
        row = df.loc[i_]
        ci, _ = letterbox_image(read_gray(row["image"]).astype(np.float32))
        gt = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
        mm = cv2.resize(bank["mask96"][id2row[row.id]], (IMG_SIZE, IMG_SIZE),
                        interpolation=cv2.INTER_NEAREST)
        ring = make_ring(mm.astype(np.uint8))
        def overlay(base, red, blue):
            o = np.dstack([base, base, base]).copy()
            o[..., 0] = np.clip(o[..., 0] + 0.5 * red, 0, 1)
            o[..., 2] = np.clip(o[..., 2] + 0.35 * blue, 0, 1)
            return o
        axes[r_, 0].imshow(ci, cmap="gray"); axes[r_, 0].set_title(f"{row.id} label={int(row.label)}")
        axes[r_, 1].imshow(overlay(ci, gt, gt * 0)); axes[r_, 1].set_title("true mask (oracle)")
        axes[r_, 2].imshow(overlay(ci, mm, ring))
        axes[r_, 2].set_title(f"pred mask+ring, p={p_q[id2row[row.id]]:.2f}, "
                              f"dice={bank['dice'][id2row[row.id]]:.2f}")
        for a in axes[r_]:
            a.axis("off")
    plt.suptitle(f"{nm}: lesion-grounded ROI vs true mask", y=1.0)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f"phase3e_qual_{nm.replace('-', '_').lower()}.png", dpi=140, bbox_inches="tight")
    plt.show()

qual_grid(DDTI_BANK, ddti, "DDTI", SEED + 23)
qual_grid(TRAIN_BANK, tnui_train, "TNUI-train", SEED + 24)

## 9. Run summary and hand-off

In [ ]:
run = {"phase": "3e - DDTI + TNUI-train external arms (pre-declared)",
       "seed": SEED, "device": str(DEVICE),
       "protocol": {"threshold_source": "TN5000 val (raw-raw)", "temperature_source": "TN5000 val",
                    "support_episode": "same as Phase 3 (verified vs run_summary.json)",
                    "oracle": "true masks per dataset",
                    "heads_provenance": HEADS_SRC,
                    "gates": {"replication_tol": _hetol, "gate_ok": bool(GATE_OK),
                              "drift_test": {m: float(v) for m, v in drift_t.items()},
                              "drift_val_descriptive": drift_v},
                    "ddti_labels": "XML diagnoses; literal words > numeric 1/3; unknown excluded; malignant-if-any",
                    "declaration": "both arms + families specified before evaluation"},
       "cohorts": {nm: {"n": int(len(Y[nm])), "malignant": int(Y[nm].sum()),
                        "mean_dice": float(np.mean(BANKS[nm]["dice"]))} for nm in BANKS},
       "audits": aud,
       "tables": {"ddti_main": "phase3e_ddti_main_table.csv", "ddti_paired": "phase3e_ddti_paired.csv",
                  "tnuitrain_main": "phase3e_tnuitrain_main_table.csv",
                  "tnuitrain_paired": "phase3e_tnuitrain_paired.csv",
                  "evidence_auroc": "phase3e_evidence_auroc.csv",
                  "evidence_sens": "phase3e_evidence_sens.csv",
                  "sign_consistency": "phase3e_sign_consistency.csv"}}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts:")
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size/1024:9.1f} KB")

## 10. What to send back (one by one)
1. §1 manifest: DDTI pairs/XML-labelled/usable **(esp. benign count)**, TNUI-train usable,
   all audit lines (expect ddti within-dups=2);
2. §4 bank table (4 cohorts: n, mean Dice vs refs 0.678 DDTI / 0.708 TNUI, empty rates);
3. §5 heads provenance + ERM prob counts;
4. §6 replication-gate dicts (test tol-gated; val descriptive vs locked 3d) + ERM sanity —
   then the DDTI and TNUI-train main tables;
5. §7 E1–E5 and F1–F5 paired tables + the two evidence matrices + sign consistency;
6. §8 qualitative descriptions (both grids).

**Reading:** DDTI decides whether 3c's rank-transfer pattern was TNUI-test-specific
(E2 negative-and-significant / E4 significant would support a general grounding advantage
in rank; E ≈ 0 across the board would confirm cohort-dependence). TNUI-train decides
whether val is the odd split *within* TNUI (train ≈ test ⇒ val anomaly ⇒ likely label
noise documented as limitation; train ≈ val ⇒ TNUI-external association generally weak
except test ⇒ temper all TNUI conclusions). Either outcome writes straight into the
externalisation section as measured evidence, null or not.